# The Simpsons Character Dialogue Analytics

## Data Visualization - Part 2

**Author:** Alexis Vendrix

**Date:** May 31, 2026

This notebook explores character dialogue patterns across episodes and seasons of The Simpsons. The part 2 focuses on the characters: Who speaks the most in general or in a episode? How does the script evolve over the seasons? 

### Libraries

In [2]:
import pandas as pd
import numpy as np
import altair as alt
import os

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

## 1. Data Loading

In [ ]:
script_df = pd.read_csv('../data/simpsons_script_lines.csv', low_memory=False)
episodes_df = pd.read_csv('../data/simpsons_episodes_clean.csv')

In [4]:
script_df.head()

,id,episode_id,number,raw_text,timestamp_in_ms,speaking_line,character_id,location_id,raw_character_text,raw_location_text,spoken_words,normalized_text,word_count
0,9549,32,209,"Miss Hoover: No, actually, it was a little of ...",848000,true,464,3.0,Miss Hoover,Springfield Elementary School,"No, actually, it was a little of both. Sometim...",no actually it was a little of both sometimes ...,31
1,9550,32,210,Lisa Simpson: (NEAR TEARS) Where's Mr. Bergstrom?,856000,true,9,3.0,Lisa Simpson,Springfield Elementary School,Where's Mr. Bergstrom?,wheres mr bergstrom,3
2,9551,32,211,Miss Hoover: I don't know. Although I'd sure l...,856000,true,464,3.0,Miss Hoover,Springfield Elementary School,I don't know. Although I'd sure like to talk t...,i dont know although id sure like to talk to h...,22
3,9552,32,212,Lisa Simpson: That life is worth living.,864000,true,9,3.0,Lisa Simpson,Springfield Elementary School,That life is worth living.,that life is worth living,5
4,9553,32,213,Edna Krabappel-Flanders: The polls will be ope...,864000,true,40,3.0,Edna Krabappel-Flanders,Springfield Elementary School,The polls will be open from now until the end ...,the polls will be open from now until the end ...,33


## 2. Data processing

In [9]:
# Drop unused columns
cols_to_drop = ['id', 'character_id', 'location_id', 'raw_text','raw_location_text', 'normalized_text']
script_df.drop(columns=[col for col in cols_to_drop if col in script_df.columns], inplace=True)

# Episode mapping (season + number_in_season)
episode_mapping = episodes_df[['number_in_series', 'number_in_season', 'season']].rename(
    columns={'number_in_series': 'episode_id'}
)

# Filter to speaking lines only
dialogue_df = script_df[script_df['speaking_line'] == 'true'].copy()
dialogue_df.drop(columns=['speaking_line'], inplace=True)

# Remove rows without character names or word counts
dialogue_df = dialogue_df.dropna(subset=['raw_character_text', 'word_count'])

# Type conversions
dialogue_df['word_count'] = pd.to_numeric(dialogue_df['word_count'], errors='coerce')
dialogue_df['timestamp_in_ms'] = pd.to_numeric(dialogue_df['timestamp_in_ms'], errors='coerce')
dialogue_df['sentence_count'] = 1

# Merge to add season and episode number
dialogue_df = dialogue_df.merge(
    episode_mapping[['episode_id', 'season', 'number_in_season']],
    on='episode_id',
    how='left'
)

dialogue_df = dialogue_df.dropna(subset=['season'])
dialogue_df['season'] = dialogue_df['season'].astype(int)

dialogue_df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 132110 entries, 0 to 132109
Data columns (total 9 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   episode_id          132110 non-null  int64  
 1   number              132110 non-null  int64  
 2   timestamp_in_ms     132110 non-null  int64  
 3   raw_character_text  132110 non-null  object 
 4   spoken_words        132110 non-null  object 
 5   word_count          132095 non-null  float64
 6   sentence_count      132110 non-null  int64  
 7   season              132110 non-null  int64  
 8   number_in_season    132110 non-null  int64  
dtypes: float64(1), int64(6), object(2)
memory usage: 9.1+ MB


### Top character selection
We limit this project to the top 10 characters

In [ ]:
top_char_list = (
    dialogue_df.groupby('raw_character_text')['sentence_count']
    .sum()
    .nlargest(10)
    .index
    .tolist()
)
dialogue_top = dialogue_df[dialogue_df['raw_character_text'].isin(top_char_list)]


## Timestamp imputation
I have observed missing timestamps within episode 138. An imputation is made base on the number of the sentence from other episodes.

In [ ]:
dialogue_top[dialogue_top['episode_id'] == 138].sort_values("number")

,episode_id,number,timestamp_in_ms,raw_character_text,spoken_words,word_count,sentence_count,season,number_in_season
25039,138,9,0,Homer Simpson,"Well, goodnight, son.",3.0,1,7,10
25040,138,10,0,Bart Simpson,"Um, Dad...",2.0,1,7,10
25041,138,11,0,Homer Simpson,Yeah.,1.0,1,7,10
25042,138,12,0,Bart Simpson,What is the mind? Is it just a system of impul...,16.0,1,7,10
25043,138,13,0,Homer Simpson,Relax. What is mind? No matter. What is matter...,11.0,1,7,10
...,...,...,...,...,...,...,...,...,...
25216,138,200,0,C. Montgomery Burns,Stricken I lurched forth in search of aide.,8.0,1,7,10
25217,138,201,0,C. Montgomery Burns,"But finding only slack-jawed gawkers, I gave u...",13.0,1,7,10
25218,138,202,0,Lisa Simpson,"Then, with your last ounce of strength, you po...",15.0,1,7,10
25219,138,203,0,Marge Simpson,"Well, I'm just relieved that Homer's safe and ...",18.0,1,7,10


In [20]:
# Step 1: Replace 0 with NaN
dialogue_top['timestamp_in_ms'] = dialogue_top['timestamp_in_ms'].replace(0, np.nan)

# Step 2: Sort by episode and line number
dialogue_top = dialogue_top.sort_values(['episode_id', 'number'])

# Step 3: Interpolate within each episode (handles partial gaps)
dialogue_top['timestamp_in_ms'] = (
    dialogue_top.groupby('episode_id')['timestamp_in_ms']
    .transform(lambda x: x.interpolate(method='linear'))
)

# Step 4: For episodes still fully NaN, impute using median timestamp per line number
median_by_number = dialogue_top.groupby('number')['timestamp_in_ms'].median()

dialogue_top['timestamp_in_ms'] = dialogue_top.apply(
    lambda row: median_by_number.get(row['number'], 0)
    if pd.isna(row['timestamp_in_ms'])
    else row['timestamp_in_ms'],
    axis=1
)

dialogue_top['timestamp_in_ms'] = dialogue_top['timestamp_in_ms'].fillna(0).astype(int)



/var/folders/v5/jx4rcwvs44bd_b45d0yvr_8w0000gn/T/ipykernel_90400/1811290950.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dialogue_top['timestamp_in_ms'] = dialogue_top['timestamp_in_ms'].replace(0, np.nan)


In [ ]:
dialogue_top["timestamp_min"] = (dialogue_top["timestamp_in_ms"] / 1000).round(0).astype(int)
dialogue_top[dialogue_top['episode_id'] == 1].sort_values("number")

,episode_id,number,timestamp_in_ms,raw_character_text,spoken_words,word_count,sentence_count,season,number_in_season,timestamp_min
123819,1,2,8000,Marge Simpson,"Ooo, careful, Homer.",3.0,1,1,1,8
123820,1,3,10000,Homer Simpson,There's no time to be careful.,6.0,1,1,1,10
123821,1,4,10000,Homer Simpson,We're late.,2.0,1,1,1,10
123822,1,7,24000,Marge Simpson,"Sorry, Excuse us. Pardon me...",5.0,1,1,1,24
123823,1,8,26000,Homer Simpson,"Hey, Norman. How's it going? So you got dragge...",21.0,1,1,1,26
...,...,...,...,...,...,...,...,...,...,...
124150,1,372,1358000,Marge Simpson,"Take it, Homer!",3.0,1,1,1,1358
124151,1,373,1360000,Homer Simpson,RUDOLPH WITH YOUR NOSE... OVER HERE... / SO YO...,13.0,1,1,1,1360
124152,1,374,1365000,Marge Simpson,"Oh, Homer.",2.0,1,1,1,1365
124153,1,375,1366000,Lisa Simpson,THEN ALL THE REINDEER LOVED HIM / AS THEY SHOU...,22.0,1,1,1,1366


### Outliers handling for count of words
Some number of words for a sentence were too high (e.g. 409 000 words) and had to be treated. 

In [23]:
bounds_by_char = {}

for char in top_char_list:
    char_data = dialogue_top[dialogue_top['raw_character_text'] == char]['word_count']
    
    Q1 = char_data.quantile(0.25)
    Q3 = char_data.quantile(0.75)
    IQR = Q3 - Q1
    upper_bound = Q3 + 15 * IQR
    
    bounds_by_char[char] = {
        'Q1': Q1,
        'Q3': Q3,
        'IQR': IQR,
        'upper_bound': upper_bound,
        'mean': char_data.mean(),
        'max': char_data.max()
    }

print("\n" + "=" * 80)
print("UPPER OUTLIERS (by character-specific bounds)")
print("=" * 80)

upper_outliers_list = []

for char in top_char_list:
    char_data = dialogue_top[dialogue_top['raw_character_text'] == char]
    upper_bound = bounds_by_char[char]['upper_bound']
    
    outliers = char_data[char_data['word_count'] > upper_bound]
    
    for idx, row in outliers.iterrows():
        upper_outliers_list.append({
            'episode_id': row['episode_id'],
            'season': row['season'],
            'character': char,
            'word_count': row['word_count'],
            'upper_bound': upper_bound,
            'exceeds_by': row['word_count'] - upper_bound
        })
        
        
# SUMMARY TABLE
print("\n" + "=" * 80)
print("CHARACTER BOUNDS SUMMARY:")
print("-" * 80)

bounds_summary = pd.DataFrame(bounds_by_char).T.reset_index()
bounds_summary.columns = ['character', 'Q1', 'Q3', 'IQR', 'upper_bound', 'mean', 'max']
bounds_summary = bounds_summary[['character', 'mean', 'max', 'Q1', 'Q3', 'IQR', 'upper_bound']]
print(bounds_summary.to_string(index=False))

upper_outliers_top = pd.DataFrame(upper_outliers_list).sort_values('word_count', ascending=False)

print(f"\nTotal upper outliers: {len(upper_outliers_top)} lines\n")

# OUTLIERS BY CHARACTER
print("OUTLIERS BY CHARACTER (using character-specific IQR):")
print("-" * 80)

char_summary = upper_outliers_top.groupby('character').agg({
    'word_count': ['count', 'max', 'mean'],
    'exceeds_by': 'mean'
}).reset_index()

char_summary.columns = ['character', 'outlier_count', 'max_words', 'avg_words', 'avg_exceeds_by']
char_summary = char_summary.sort_values('outlier_count', ascending=False)

print(char_summary.to_string(index=False))

# OUTLIERS BY EPISODE
print("\n" + "=" * 80)
print("PROBLEM EPISODES:")
print("-" * 80)

episode_summary = upper_outliers_top.groupby('episode_id').apply(
    lambda x: pd.Series({
        'season': x['season'].iloc[0],
        'num_outliers': len(x),
        'characters': ', '.join(x['character'].unique()),
        'max_words': x['word_count'].max()
    })
).reset_index().sort_values('max_words', ascending=False)

print(episode_summary.to_string(index=False))



UPPER OUTLIERS (by character-specific bounds)

CHARACTER BOUNDS SUMMARY:
--------------------------------------------------------------------------------
          character      mean       max  Q1   Q3  IQR  upper_bound
      Homer Simpson 24.371480  409000.0 4.0 13.0  9.0        148.0
      Marge Simpson 96.248484 1145000.0 4.0 13.0  9.0        148.0
       Bart Simpson 16.684316  108000.0 4.0 11.0  7.0        116.0
       Lisa Simpson 33.728666  147000.0 4.0 13.0  9.0        148.0
C. Montgomery Burns 11.730483      91.0 5.0 16.0 11.0        181.0
        Moe Szyslak 11.696333     100.0 5.0 16.0 11.0        181.0
    Seymour Skinner 11.773222      67.0 5.0 16.0 11.0        181.0
       Ned Flanders 11.137093      74.0 5.0 15.0 10.0        165.0
     Grampa Simpson 10.796901     122.0 4.0 14.0 10.0        164.0
       Chief Wiggum 11.131403      67.0 5.0 15.0 10.0        165.0

Total upper outliers: 5 lines

OUTLIERS BY CHARACTER (using character-specific IQR):
----------------------

/var/folders/v5/jx4rcwvs44bd_b45d0yvr_8w0000gn/T/ipykernel_90400/2034517704.py:76: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  episode_summary = upper_outliers_top.groupby('episode_id').apply(


### Outliers substitution

In [ ]:
# REPLACE OUTLIERS using bounds_by_char from previous step
dialogue_cleaned = dialogue_top.copy()

print("=" * 80)
print("REPLACING OUTLIERS WITH CHARACTER MEAN")
print("=" * 80)

replacements_made = []

for char in top_char_list:
    char_mask = dialogue_cleaned['raw_character_text'] == char
    
    # Reuse bounds from previous step
    upper_bound = bounds_by_char[char]['upper_bound']
    char_mean = bounds_by_char[char]['mean']
    
    # Find and replace outliers
    outlier_mask = char_mask & (dialogue_cleaned['word_count'] > upper_bound)
    num_outliers = outlier_mask.sum()
    
    if num_outliers > 0:
        old_values = dialogue_cleaned[outlier_mask]['word_count'].copy()
        dialogue_cleaned.loc[outlier_mask, 'word_count'] = char_mean
        
        replacements_made.append({
            'character': char,
            'num_replaced': num_outliers,
            'max_old_value': old_values.max(),
        })
        
        print(f"{char}: {num_outliers} outliers → mean={char_mean:.1f}")

# Summary
print("\n" + "=" * 80)
if replacements_made:
    summary_df = pd.DataFrame(replacements_made)
    print(f"Total outliers replaced: {summary_df['num_replaced'].sum()}\n")
    print(summary_df.to_string(index=False))
else:
    print("No outliers found")

print("\n✓ dialogue_cleaned ready for visualizations")



REPLACING OUTLIERS WITH CHARACTER MEAN
Homer Simpson: 1 outliers → mean=24.4
Marge Simpson: 1 outliers → mean=96.2
Bart Simpson: 1 outliers → mean=16.7
Lisa Simpson: 2 outliers → mean=33.7

Total outliers replaced: 5

    character  num_replaced  max_old_value
Homer Simpson             1       409000.0
Marge Simpson             1      1145000.0
 Bart Simpson             1       108000.0
 Lisa Simpson             2       147000.0

✓ dialogue_cleaned ready for visualizations


### Export clean dataset to CSV

In [ ]:
dialogue_cleaned.to_csv('dialogue_cleaned.csv', index=False)

,episode_id,number,timestamp_in_ms,raw_character_text,spoken_words,word_count,sentence_count,season,number_in_season,timestamp_min
123819,1,2,8000,Marge Simpson,"Ooo, careful, Homer.",3.0,1,1,1,8
123820,1,3,10000,Homer Simpson,There's no time to be careful.,6.0,1,1,1,10
123821,1,4,10000,Homer Simpson,We're late.,2.0,1,1,1,10
123822,1,7,24000,Marge Simpson,"Sorry, Excuse us. Pardon me...",5.0,1,1,1,24
123823,1,8,26000,Homer Simpson,"Hey, Norman. How's it going? So you got dragge...",21.0,1,1,1,26
123824,1,9,34000,Homer Simpson,Pardon my galoshes.,3.0,1,1,1,34
123825,1,10,44000,Seymour Skinner,"Wasn't that wonderful? And now, ""Santas of Man...",17.0,1,1,1,44
123826,1,11,55000,Marge Simpson,Oh... Lisa's class.,3.0,1,1,1,55
123830,1,15,98000,Homer Simpson,"Oh, it's Lisa. That's ours.",5.0,1,1,1,98
123831,1,16,114000,Seymour Skinner,The fourth grade will now favor us with a melo...,14.0,1,1,1,114


## 2. Theme & Character Colors

I define a custom Altair theme and establish fixed character colors that will be consistent across all visualizations.

In [ ]:
# Register custom Altair theme
@alt.theme.register("simpsons_dialogue", enable=True)
def simpsons_dialogue_theme():
    return alt.theme.ThemeConfig(
        #background="transparent",
        view={"stroke": "transparent"},
        range={
            "category": ["#FDB750", "#00F502", "#FF7F00", "#FF1493", "#8B0000", 
                         "#6495ED", "#DAA520", "#FF69B4", "#4169E1", "#9370DB"],
        },
        title={"font": "sans-serif", "color": "#333333", "fontSize": 18, "anchor": "start"},
        axis={
            "labelColor": "#333333", "titleColor": "#333333", "gridColor": "#D0D0D0",
            "labelFontSize": 12, "titleFontSize": 14,
        },
    )

# Character color mapping - TOP 10 CHARACTERS ONLY (fixed colors for consistency)
character_colors = {
    'Homer Simpson': '#FFD90F',        # Iconic Yellow
    'Marge Simpson': '#009DDC',        # Tall Blue Hair
    'Bart Simpson': '#F14E28',         # Signature Red Shirt
    'Lisa Simpson': '#FF69B4',         # Pink Dress
    'C. Montgomery Burns': '#93C47D',  # Sickly Green/Suit
    'Moe Szyslak': '#493D26',          # Apron Brown
    'Seymour Skinner': '#445491',      # Suit Blue
    'Ned Flanders': '#709139',         # Sweater Green
    'Grampa Simpson': '#EEDAB0',       # Beige Cardigan
    'Chief Wiggum': '#32499F',         # Uniform Blue
}



## 3. Visualizations & design decisions

The final deliverable is the Streamlit app (`Vendrix_Alexis_streamlit.py`).
This section comment the design of each view and and graphs but are not representative of the final design because some interaction where made within `Streamlit`. 

A global sidebar **metric toggle (Words / Sentences)** drives every chart, so all graphs share one implementation. All views reuse one fixed *character colour* map (`character_colors`) so a colour always means the same character.


### Aggregation and top 10 characters selection

In [ ]:
character_stats = dialogue_cleaned.groupby('raw_character_text').agg({
    'word_count': 'sum',
    'sentence_count': 'sum',
    'episode_id': 'nunique'
}).reset_index()

character_stats.columns = ['character_name', 'total_words', 'total_sentences', 'episodes_appeared']
character_stats['words_per_sentence'] = (character_stats['total_words'] / character_stats['total_sentences']).round(2)
character_stats = character_stats.sort_values('total_sentences', ascending=False)

top_characters = character_stats['character_name'].head(10).to_list()

### Q1 — Q5 - Which characters issue the most words/senteces, and how are they distributed?

A horizontal bar chart of the top-10 characters by total words (or sentenceds) was choosen, sorted descending. The bars encode magnitude making it easy for ranking and comparison. 

The horizontal layout leaves room for full character names without rotated labels.

The y-axis title is dropped and the colour legend is suppressed because the labelled bars already identify each character.

**Interaction.** Clicking bars toggles a multi-character selection (`toggle='true'`). The same selection cross-filters the season chart (Q2), so a chosen character is highlighted there too.


In [ ]:
color_scale = alt.Scale(domain=list(character_colors.keys()),
                        range=list(character_colors.values()))

char_words = character_stats[character_stats['character_name'].isin(top_characters)]
char_click = alt.selection_point(fields=['character_name'], on='click', toggle='true')

q1 = (
    alt.Chart(char_words)
    .mark_bar()
    .encode(
        y=alt.Y('character_name:N', sort='-x', title=None),
        x=alt.X('total_words:Q', title='Words Spoken'),
        color=alt.Color('character_name:N', scale=color_scale, legend=None),
        opacity=alt.condition(char_click, alt.value(1), alt.value(0.3)),
        tooltip=['character_name:N', 'total_words:Q', 'episodes_appeared:Q'],
    )
    .add_params(char_click)
    .properties(width=520, height=320, title='Top characters by words spoken')
)
q1


alt.Chart(...)

#### Q5 — Switch between words and sentences

In the Streamlit app this is not a separate chart: a sidebar radio switches the global metric between *Word Count* and *Sentence Count*, and every view re-renders on the chosen measure. This switch allows a rapid change between measure but make it harder to comparare the words count metric with the sentences metric.

In [60]:
char_sent = character_stats[character_stats['character_name'].isin(top_characters)]
char_click = alt.selection_point(fields=['character_name'], on='click', toggle='true')

q5 = (
    alt.Chart(char_sent)
    .mark_bar()
    .encode(
        y=alt.Y('character_name:N', sort='-x', title=None),
        x=alt.X('total_sentences:Q', title='Speaking Lines (sentences)'),
        color=alt.Color('character_name:N', scale=color_scale, legend=None),
        opacity=alt.condition(char_click, alt.value(1), alt.value(0.3)),
        tooltip=['character_name:N', 'total_sentences:Q', 'words_per_sentence:Q'],
    )
    .add_params(char_click)
    .properties(width=520, height=320, title='Top characters by speaking lines')
)
q5


alt.Chart(...)

### Q2 — How have word numbers evolved across seasons?

A multi-series line chart was choosen (x = season, y = words, one line per character). Lines chart are is the recommended design to encode the evolution of a metric over time. It makes trends and crossovers easy to understand.

Ten overlapping lines are unreadable on their own. To solve this, I added an interaction to focus on choosen characters. The Q1 selection drives opacity — selected characters stay at full opacity, so a chosen trajectory stand out. Below, the bar and line are shown together sharing one selection to demonstrate it.


In [61]:
char_season = (
    dialogue_cleaned[dialogue_cleaned['raw_character_text'].isin(top_characters)]
    .groupby(['season', 'raw_character_text'])['word_count'].sum()
    .reset_index()
    .rename(columns={'raw_character_text': 'character_name', 'word_count': 'words'})
)

bar = (
    alt.Chart(char_season.groupby('character_name', as_index=False)['words'].sum())
    .mark_bar()
    .encode(
        y=alt.Y('character_name:N', sort='-x', title=None),
        x=alt.X('words:Q', title='Words Spoken'),
        color=alt.Color('character_name:N', scale=color_scale, legend=None),
        opacity=alt.condition(char_click, alt.value(1), alt.value(0.3)),
        tooltip=['character_name:N', 'words:Q'],
    )
    .add_params(char_click)
    .properties(width=280, height=360, title='Click a character')
)

line = (
    alt.Chart(char_season)
    .mark_line(point={'size': 50})
    .encode(
        x=alt.X('season:Q', title='Season', axis=alt.Axis(format='d')),
        y=alt.Y('words:Q', title='Words per season'),
        color=alt.Color('character_name:N', scale=color_scale, legend=None),
        opacity=alt.condition(char_click, alt.value(1), alt.value(0.1)),
        tooltip=['character_name:N', 'season:Q', 'words:Q'],
    )
    .properties(width=520, height=360, title='Words spoken over seasons')
)

(bar | line)


alt.HConcatChart(...)

### Q3 — Compare a pair of characters, for a selected season *and* a selected episode

For the episode selection, a **heatmap** (x = episode, y = season, colour = total words) is used as the selector. It shows every episode at once and reveals which are with the most words/sentences spoken. Clicking a cell sets *both* the season and the episode in a single gesture, minimising the interactions needed — and drives two grouped bar charts: the **season total**  and the **episode total**. Reading top-down gives a season to episode drill-down. Bar-charts were use to compare easily the two count metrics.

A single-hue *sequential* yellow→red ramp (no problem for color blind persons) is choosen. The two characters are chosen with two dependent dropdowns list within **Streamlit** where list B excludes whoever is picked in A, so you can never compare a character with itself. A default of **s01e01 / Homer vs Marge** guarantees the panel is never blank.



In [69]:
# The actual selection is made in streamlit
pair = ['Homer Simpson', 'Marge Simpson']

episode_totals = (
    dialogue_cleaned.groupby(['season', 'number_in_season'])['word_count']
    .sum().reset_index(name='total')
)
pair_episode = (
    dialogue_cleaned[dialogue_cleaned['raw_character_text'].isin(pair)]
    .groupby(['season', 'number_in_season', 'raw_character_text'])['word_count']
    .sum().reset_index()
)

pair_season = (
    dialogue_cleaned[dialogue_cleaned['raw_character_text'].isin(pair)]
    .groupby(['season', 'raw_character_text'])['word_count'].sum().reset_index()
)


episode_click = alt.selection_point(
    name='episode_click',
    fields=['season', 'number_in_season'], on='click', empty=False,
    value=[{'season': 1, 'number_in_season': 1}],
)


heatmap = (
    alt.Chart(episode_totals)
    .mark_rect() 
    .encode(
        x=alt.X('number_in_season:O', title='Episode # in season', axis=alt.Axis(labelAngle=0)),
        y=alt.Y('season:O', title='Season'),
        color=alt.Color('total:Q', title='Words', scale=alt.Scale(scheme='yelloworangered'),
                        legend=alt.Legend(orient='right')),
        opacity=alt.condition(episode_click, alt.value(1.0), alt.value(0.75)),
        stroke=alt.condition(episode_click, alt.value('#000000'), alt.value('#FFFEF9')),
        strokeWidth=alt.condition(episode_click, alt.value(2.5), alt.value(0.5)),
        tooltip=['season:O', 'number_in_season:O', 'total:Q'],
    )
    .add_params(episode_click)
    .properties(width=460, height=360, title='Click an episode')
)

# Season total: filter to the clicked season only
season_bars = (
    alt.Chart(pair_season)
    .mark_bar()
    .encode(
        x=alt.X('raw_character_text:N', title=None, axis=alt.Axis(labelAngle=0)),
        y=alt.Y('word_count:Q', title='Words spoken'),
        color=alt.Color('raw_character_text:N', scale=color_scale, legend=None),
        tooltip=['raw_character_text:N', 'word_count:Q'],
    )
    .transform_filter('datum.season == episode_click.season')
    .properties(width=180, height=360, title='Selected season: Homer vs Marge')
) 

# Episode total
episode_bars = (
    alt.Chart(pair_episode)
    .mark_bar()
    .encode(
        x=alt.X('raw_character_text:N', title=None, axis=alt.Axis(labelAngle=0)),
        y=alt.Y('word_count:Q', title='Words spoken'),
        color=alt.Color('raw_character_text:N', scale=color_scale, legend=None),
        tooltip=['raw_character_text:N', 'word_count:Q'],
    )
    .transform_filter(episode_click)
    .properties(width=180, height=360, title='Selected episode: Homer vs Marge')
)

(heatmap | season_bars | episode_bars).resolve_scale(color='independent')


alt.HConcatChart(...)

### Q4 — The same comparison, but *within* one concrete episode

A group bar chart was choosen to display the count over time for the two characters within the selected episode, with a thin **tick-rug + brush** strip underneath. 

The x-axis is the episode timeline in seconds, formatted as `Xmin Ysec`, so we can read when dialogue happens and how turn-taking unfolds.

The timestamps are converted from milliseconds to seconds and aggregated per second per character. The two characters are offset by ±0.4s so their bars sit side-by-side instead of overlapping at the same instant. 

Dragging a range on the rug zooms the detail chart's x-axis while the rug stays fixed as a stable reference. Tooltips also display the character, the metric and the exact words spoken at each moment.


In [ ]:
sel_s, sel_e = 10, 12
pair = ['Homer Simpson', 'Marge Simpson']

ep = dialogue_cleaned[
    (dialogue_cleaned['season'] == sel_s)
    & (dialogue_cleaned['number_in_season'] == sel_e)
    & (dialogue_cleaned['raw_character_text'].isin(pair))
].copy()
ep['timestamp_sec'] = (ep['timestamp_in_ms'] / 1000).round().astype(int)

tl = ep.groupby(['timestamp_sec', 'raw_character_text'])['word_count'].sum().reset_index()
offsets = {pair[0]: -0.4, pair[1]: 0.4}
tl['x_pos'] = tl['timestamp_sec'] + tl['raw_character_text'].map(offsets)

brush = alt.selection_interval(encodings=['x'])
time_expr = "floor(datum.value/60)+'min '+(datum.value%60<10?'0':'')+(datum.value%60)+'sec'"

detail = (
    alt.Chart(tl)
    .mark_bar(size=4)
    .encode(
        x=alt.X('x_pos:Q', title='Time in episode',
                scale=alt.Scale(domain=brush, nice=False),
                axis=alt.Axis(labelExpr=time_expr, labelAngle=-40)),
        y=alt.Y('word_count:Q', title='Words spoken', stack=None),
        color=alt.Color('raw_character_text:N', scale=color_scale, legend=None),
        tooltip=['raw_character_text:N', 'word_count:Q'],
    )
    .properties(width=700, height=160, title=f'S{sel_s:02d}E{sel_e:02d} — brush the rug to zoom')
)

rug = (
    alt.Chart(tl)
    .mark_tick(thickness=1, color='#666666')
    .encode(x=alt.X('timestamp_sec:Q', title=None, axis=alt.Axis(labels=False, ticks=False)))
    .add_params(brush)
    .properties(width=700, height=18)
)

alt.vconcat(detail, rug, spacing=4).resolve_scale(x='independent')


alt.VConcatChart(...)

## 4. Final visualization

The final visualization is in the Streamlit application. (`Vendrix_Alexis_streamlit.py`).

**Consistency.** One fixed character colour map is used in every chart, so a colour always denotes the same character (same colour = same meaning). The font, text, axis colours and gridlines are consistent. The page uses the Simpsons palette everywhere.

**Layout.** Related views are grouped: Q1 + Q2 share the top row (who speaks, and how that evolves). The heatmap and the Q3/Q4 drill-down sit below, going from a full serie overview towards season towards episode towards episode's timeline. 

The horizontal alignment supports side-by-side comparison and the vertical stack tells the drill-down story.

**Interaction.** Two cross-interactions were used: bar-click highlights the season lines and the heatmap-click feeds the pair bars and the timeline. The global metric toggle keeps every view on the same measure. Defaults (s01e01, Homer vs Marge) and guards (dependent dropdowns, fallbacks) ensure no view is ever blank or full of NaNs.

**Colour-blindness.** No red/green encodings, the heatmap is a single-hue gradient.
